In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import torch
import boto3
import os
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    DistilBertTokenizer,
    AutoTokenizer
)
from torch.utils.data import (
    Dataset, DataLoader
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from dotenv import load_dotenv

load_dotenv()

# Check if GPU is available
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu')
print(f"✅ Using device: {device}")
print(f"   PyTorch version: {torch.__version__}")

# Verify transformers
from transformers import __version__ as tv
print(f"   Transformers version: {tv}")

✅ Using device: cpu
   PyTorch version: 2.4.1+cu118
   Transformers version: 5.3.0


In [2]:
# Cell 2 — Load from S3
print("Loading data from AWS S3...")

s3_client = boto3.client(
    's3',
    aws_access_key_id=os.getenv(
        'AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv(
        'AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_DEFAULT_REGION')
)

bucket = os.getenv('S3_BUCKET_DATA')

for filename in ['ag_news_train.csv',
                 'ag_news_test.csv']:
    s3_client.download_file(
        bucket,
        f'raw/{filename}',
        f'../data/raw/{filename}'
    )

df_train = pd.read_csv(
    '../data/raw/ag_news_train.csv')
df_test  = pd.read_csv(
    '../data/raw/ag_news_test.csv')

print(f"✅ Train: {df_train.shape}")
print(f"✅ Test:  {df_test.shape}")

Loading data from AWS S3...
✅ Train: (120000, 5)
✅ Test:  (7600, 3)


In [3]:
# Cell 3 — Explain model choice
print("""
MODEL COMPARISON:
─────────────────────────────────────────────────
BERT-base-uncased:
  Parameters:  110 million
  Layers:      12 transformer layers
  Hidden size: 768
  Training:    Very slow on CPU
  Accuracy:    Highest

DistilBERT-base-uncased (OUR CHOICE):
  Parameters:  66 million  ← 40% smaller
  Layers:      6 transformer layers
  Hidden size: 768
  Training:    2x faster than BERT
  Accuracy:    97% of BERT performance
  How:         Knowledge distillation from BERT

RoBERTa-base:
  Parameters:  125 million
  Training:    Longer than BERT
  Accuracy:    Better than BERT
  Tradeoff:    Slower

For portfolio project on CPU/small GPU:
→ DistilBERT gives best speed/accuracy tradeoff
→ In interview: "I chose DistilBERT for its
  97% performance retention at 40% fewer
  parameters — practical for production
  where inference speed matters"
─────────────────────────────────────────────────
""")


MODEL COMPARISON:
─────────────────────────────────────────────────
BERT-base-uncased:
  Parameters:  110 million
  Layers:      12 transformer layers
  Hidden size: 768
  Training:    Very slow on CPU
  Accuracy:    Highest

DistilBERT-base-uncased (OUR CHOICE):
  Parameters:  66 million  ← 40% smaller
  Layers:      6 transformer layers
  Hidden size: 768
  Training:    2x faster than BERT
  Accuracy:    97% of BERT performance
  How:         Knowledge distillation from BERT

RoBERTa-base:
  Parameters:  125 million
  Training:    Longer than BERT
  Accuracy:    Better than BERT
  Tradeoff:    Slower

For portfolio project on CPU/small GPU:
→ DistilBERT gives best speed/accuracy tradeoff
→ In interview: "I chose DistilBERT for its
  97% performance retention at 40% fewer
  parameters — practical for production
  where inference speed matters"
─────────────────────────────────────────────────



In [4]:
# Cell 4 — Load DistilBERT tokenizer
MODEL_NAME = os.getenv(
    'MODEL_NAME', 'distilbert-base-uncased')
MAX_LENGTH = int(os.getenv('MAX_LENGTH', 128))

print(f"Loading tokenizer: {MODEL_NAME}")
print(f"Max sequence length: {MAX_LENGTH}")
print("Downloading tokenizer files...")

tokenizer = DistilBertTokenizer.from_pretrained(
    MODEL_NAME)

print(f"\n✅ Tokenizer loaded")
print(f"   Vocabulary size: "
      f"{tokenizer.vocab_size:,} tokens")
print(f"   Special tokens:")
print(f"     [CLS] = {tokenizer.cls_token_id}")
print(f"     [SEP] = {tokenizer.sep_token_id}")
print(f"     [PAD] = {tokenizer.pad_token_id}")
print(f"     [UNK] = {tokenizer.unk_token_id}")
# ```

# **Special tokens explained:**
# ```
# [CLS] = Classification token
#         Added at start of every sequence
#         BERT uses this token's representation
#         for classification tasks
#         Like saying "START — classify what follows"

# [SEP] = Separator token
#         Added at end of every sequence
#         Marks where the text ends
#         Like saying "END OF DOCUMENT"

# [PAD] = Padding token
#         Added to shorter sequences in a batch
#         So all sequences same length
#         Attention mask = 0 for padding tokens

# [UNK] = Unknown token
#         Rarely used in DistilBERT because
#         WordPiece handles most words

Loading tokenizer: distilbert-base-uncased
Max sequence length: 128



✅ Tokenizer loaded
   Vocabulary size: 30,522 tokens
   Special tokens:
     [CLS] = 101
     [SEP] = 102
     [PAD] = 0
     [UNK] = 100


In [6]:
# Cell 5 — Show tokenization in detail
print("=== TOKENIZATION EXAMPLES ===\n")

sample_texts = [
    "Apple reported record iPhone sales.",
    "The team won the championship game.",
    "Scientists discovered a new exoplanet.",
    "Short."  # Very short text
]

for text in sample_texts:
    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    tokens = tokenizer.convert_ids_to_tokens(
        encoding['input_ids'][0])
    non_pad = [t for t in tokens
               if t != '[PAD]']

    print(f"Text: '{text}'")
    print(f"Tokens: {non_pad}")
    print(f"Input IDs (first 10): "
          f"{encoding['input_ids'][0][:10].tolist()}")
    print(f"Attention mask sum: "
          f"{encoding['attention_mask'][0].sum().item()} "
          f"real tokens")
    print(f"Padding tokens: "
          f"{(encoding['attention_mask'][0]==0).sum().item()}")
    print()

=== TOKENIZATION EXAMPLES ===

Text: 'Apple reported record iPhone sales.'
Tokens: ['[CLS]', 'apple', 'reported', 'record', 'iphone', 'sales', '.', '[SEP]']
Input IDs (first 10): [101, 6207, 2988, 2501, 18059, 4341, 1012, 102, 0, 0]
Attention mask sum: 8 real tokens
Padding tokens: 120

Text: 'The team won the championship game.'
Tokens: ['[CLS]', 'the', 'team', 'won', 'the', 'championship', 'game', '.', '[SEP]']
Input IDs (first 10): [101, 1996, 2136, 2180, 1996, 2528, 2208, 1012, 102, 0]
Attention mask sum: 9 real tokens
Padding tokens: 119

Text: 'Scientists discovered a new exoplanet.'
Tokens: ['[CLS]', 'scientists', 'discovered', 'a', 'new', 'ex', '##op', '##lane', '##t', '.', '[SEP]']
Input IDs (first 10): [101, 6529, 3603, 1037, 2047, 4654, 7361, 20644, 2102, 1012]
Attention mask sum: 11 real tokens
Padding tokens: 117

Text: 'Short.'
Tokens: ['[CLS]', 'short', '.', '[SEP]']
Input IDs (first 10): [101, 2460, 1012, 102, 0, 0, 0, 0, 0, 0]
Attention mask sum: 4 real tokens
Padding 

In [7]:
# Cell 6 — Text cleaning
# Minimal cleaning as EDA showed data is clean
import re

def clean_text(text):
    """
    Light cleaning for AG News.
    EDA showed: no HTML, few URLs, mostly clean.
    BERT handles numbers and punctuation natively.
    """
    if not isinstance(text, str):
        return ""

    # Remove any residual HTML
    text = re.sub(r'<[^>]+>', ' ', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove very long number sequences
    # (e.g. phone numbers, IDs)
    text = re.sub(r'\b\d{10,}\b', '[NUM]', text)

    return text

# Test cleaning
test_samples = df_train['text'].head(5).tolist()
print("=== CLEANING EXAMPLES ===")
for original in test_samples:
    cleaned = clean_text(original)
    changed = original != cleaned
    status  = "CHANGED" if changed else "unchanged"
    print(f"  [{status}] "
          f"{original[:60]}...")

# Apply to both splits
df_train['text_clean'] = df_train['text'].apply(
    clean_text)
df_test['text_clean']  = df_test['text'].apply(
    clean_text)

# Verify no empty texts
empty_train = (df_train['text_clean']
               .str.len() == 0).sum()
empty_test  = (df_test['text_clean']
               .str.len() == 0).sum()

print(f"\n✅ Cleaning complete")
print(f"   Empty texts in train: {empty_train}")
print(f"   Empty texts in test:  {empty_test}")

=== CLEANING EXAMPLES ===
  [unchanged] Wall St. Bears Claw Back Into the Black (Reuters) Reuters - ...
  [unchanged] Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters ...
  [unchanged] Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - So...
  [unchanged] Iraq Halts Oil Exports from Main Southern Pipeline (Reuters)...
  [unchanged] Oil prices soar to all-time record, posing new menace to US ...

✅ Cleaning complete
   Empty texts in train: 0
   Empty texts in test:  0


In [8]:
# Cell 7 — Train/validation split
# Keep original test set untouched

X_train_full = df_train['text_clean'].values
y_train_full = df_train['label'].values

X_train, X_val, y_train, y_val = \
    train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.1,      # 10% validation
        random_state=42,
        stratify=y_train_full
    )

X_test = df_test['text_clean'].values
y_test = df_test['label'].values

print(f"Training samples:   {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")
print(f"Test samples:       {len(X_test):,}")

# Verify class balance in each split
for name, y in [("Train", y_train),
                ("Val",   y_val),
                ("Test",  y_test)]:
    unique, counts = np.unique(
        y, return_counts=True)
    pcts = counts / len(y) * 100
    print(f"\n{name} distribution:")
    for u, c, p in zip(unique, counts, pcts):
        label = ['World','Sports',
                 'Business','Sci/Tech'][u]
        print(f"  {label:<12}: "
              f"{c:,} ({p:.1f}%)")

Training samples:   108,000
Validation samples: 12,000
Test samples:       7,600

Train distribution:
  World       : 27,000 (25.0%)
  Sports      : 27,000 (25.0%)
  Business    : 27,000 (25.0%)
  Sci/Tech    : 27,000 (25.0%)

Val distribution:
  World       : 3,000 (25.0%)
  Sports      : 3,000 (25.0%)
  Business    : 3,000 (25.0%)
  Sci/Tech    : 3,000 (25.0%)

Test distribution:
  World       : 1,900 (25.0%)
  Sports      : 1,900 (25.0%)
  Business    : 1,900 (25.0%)
  Sci/Tech    : 1,900 (25.0%)


In [9]:
# Cell 8 — Custom Dataset class
class DocumentDataset(Dataset):
    """
    PyTorch Dataset for AG News documents.

    Wraps tokenized text so DataLoader can
    efficiently batch and shuffle data during
    BERT fine-tuning.
    """

    def __init__(self, texts, labels,
                 tokenizer, max_length):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text  = str(self.texts[idx])
        label = int(self.labels[idx])

        # Tokenize single document
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding[
                'input_ids'].flatten(),
            'attention_mask': encoding[
                'attention_mask'].flatten(),
            'labels': torch.tensor(
                label, dtype=torch.long)
        }

# Test the dataset
print("Testing DocumentDataset...")
sample_dataset = DocumentDataset(
    X_train[:5], y_train[:5],
    tokenizer, MAX_LENGTH
)

sample_item = sample_dataset[0]
print(f"\n✅ Dataset working correctly")
print(f"   input_ids shape:      "
      f"{sample_item['input_ids'].shape}")
print(f"   attention_mask shape: "
      f"{sample_item['attention_mask'].shape}")
print(f"   label:                "
      f"{sample_item['labels'].item()}")

Testing DocumentDataset...

✅ Dataset working correctly
   input_ids shape:      torch.Size([128])
   attention_mask shape: torch.Size([128])
   label:                1


In [10]:
# Cell 9 — Build DataLoaders
BATCH_SIZE = int(os.getenv('BATCH_SIZE', 16))

# Create datasets
train_dataset = DocumentDataset(
    X_train, y_train, tokenizer, MAX_LENGTH)
val_dataset   = DocumentDataset(
    X_val,   y_val,   tokenizer, MAX_LENGTH)
test_dataset  = DocumentDataset(
    X_test,  y_test,  tokenizer, MAX_LENGTH)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,         # shuffle each epoch
    num_workers=0,        # 0 for Windows
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,        # never shuffle eval
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"✅ DataLoaders created")
print(f"   Training batches:   "
      f"{len(train_loader):,}")
print(f"   Validation batches: "
      f"{len(val_loader):,}")
print(f"   Test batches:       "
      f"{len(test_loader):,}")
print(f"   Batch size:         {BATCH_SIZE}")

# Test one batch
batch = next(iter(train_loader))
print(f"\n=== SAMPLE BATCH ===")
print(f"   input_ids shape:      "
      f"{batch['input_ids'].shape}")
print(f"   attention_mask shape: "
      f"{batch['attention_mask'].shape}")
print(f"   labels shape:         "
      f"{batch['labels'].shape}")
print(f"   labels in batch:      "
      f"{batch['labels'].tolist()}")

✅ DataLoaders created
   Training batches:   6,750
   Validation batches: 750
   Test batches:       475
   Batch size:         16

=== SAMPLE BATCH ===
   input_ids shape:      torch.Size([16, 128])
   attention_mask shape: torch.Size([16, 128])
   labels shape:         torch.Size([16])
   labels in batch:      [1, 3, 2, 1, 1, 0, 1, 0, 3, 2, 3, 2, 0, 0, 0, 0]


In [11]:
# Cell 10 — Tokenize entire dataset
# and save as tensors for faster loading
print("Tokenizing full datasets...")
print("This may take 5-10 minutes...")

def tokenize_dataset(texts, tokenizer,
                     max_length, batch_size=512):
    """Tokenize all texts efficiently"""
    all_input_ids      = []
    all_attention_mask = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size].tolist()

        encodings = tokenizer(
            batch_texts,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        all_input_ids.append(
            encodings['input_ids'])
        all_attention_mask.append(
            encodings['attention_mask'])

        if (i // batch_size) % 10 == 0:
            print(f"  Processed "
                  f"{min(i+batch_size, len(texts)):,}"
                  f" / {len(texts):,}")

    return (torch.cat(all_input_ids),
            torch.cat(all_attention_mask))

# Tokenize all splits
print("\nTokenizing training set...")
train_ids, train_mask = tokenize_dataset(
    X_train, tokenizer, MAX_LENGTH)

print("\nTokenizing validation set...")
val_ids, val_mask = tokenize_dataset(
    X_val, tokenizer, MAX_LENGTH)

print("\nTokenizing test set...")
test_ids, test_mask = tokenize_dataset(
    X_test, tokenizer, MAX_LENGTH)

print(f"\n✅ Tokenization complete")
print(f"   train_ids shape: {train_ids.shape}")
print(f"   val_ids shape:   {val_ids.shape}")
print(f"   test_ids shape:  {test_ids.shape}")

Tokenizing full datasets...
This may take 5-10 minutes...

Tokenizing training set...
  Processed 512 / 108,000
  Processed 5,632 / 108,000
  Processed 10,752 / 108,000
  Processed 15,872 / 108,000
  Processed 20,992 / 108,000
  Processed 26,112 / 108,000
  Processed 31,232 / 108,000
  Processed 36,352 / 108,000
  Processed 41,472 / 108,000
  Processed 46,592 / 108,000
  Processed 51,712 / 108,000
  Processed 56,832 / 108,000
  Processed 61,952 / 108,000
  Processed 67,072 / 108,000
  Processed 72,192 / 108,000
  Processed 77,312 / 108,000
  Processed 82,432 / 108,000
  Processed 87,552 / 108,000
  Processed 92,672 / 108,000
  Processed 97,792 / 108,000
  Processed 102,912 / 108,000
  Processed 108,000 / 108,000

Tokenizing validation set...
  Processed 512 / 12,000
  Processed 5,632 / 12,000
  Processed 10,752 / 12,000

Tokenizing test set...
  Processed 512 / 7,600
  Processed 5,632 / 7,600

✅ Tokenization complete
   train_ids shape: torch.Size([108000, 128])
   val_ids shape:   tor

In [12]:
# Cell 11 — Save locally and upload to S3
import pickle

os.makedirs('../data/processed', exist_ok=True)

# Save tensors locally
torch.save({
    'input_ids':      train_ids,
    'attention_mask': train_mask,
    'labels':         torch.tensor(y_train)
}, '../data/processed/train_tensors.pt')

torch.save({
    'input_ids':      val_ids,
    'attention_mask': val_mask,
    'labels':         torch.tensor(y_val)
}, '../data/processed/val_tensors.pt')

torch.save({
    'input_ids':      test_ids,
    'attention_mask': test_mask,
    'labels':         torch.tensor(y_test)
}, '../data/processed/test_tensors.pt')

# Save tokenizer locally
tokenizer.save_pretrained('../models/tokenizer')

# Save preprocessing config
prep_config = {
    'model_name':         MODEL_NAME,
    'max_length':         MAX_LENGTH,
    'batch_size':         BATCH_SIZE,
    'train_samples':      int(len(X_train)),
    'val_samples':        int(len(X_val)),
    'test_samples':       int(len(X_test)),
    'num_classes':        4,
    'label_map':          {
        '0': 'World', '1': 'Sports',
        '2': 'Business', '3': 'Sci/Tech'
    },
    'tokenizer_type':     'DistilBertTokenizer',
    'vocab_size':         tokenizer.vocab_size,
    'special_tokens':     {
        'cls': tokenizer.cls_token_id,
        'sep': tokenizer.sep_token_id,
        'pad': tokenizer.pad_token_id
    }
}

with open('../data/processed/prep_config.json',
          'w') as f:
    json.dump(prep_config, f, indent=4)

print("✅ Saved locally")

# Upload processed tensors to S3
print("\nUploading to S3...")

for local_file, s3_key in [
    ('../data/processed/train_tensors.pt',
     'processed/train_tensors.pt'),
    ('../data/processed/val_tensors.pt',
     'processed/val_tensors.pt'),
    ('../data/processed/test_tensors.pt',
     'processed/test_tensors.pt'),
    ('../data/processed/prep_config.json',
     'processed/prep_config.json'),
]:
    s3_client.upload_file(
        local_file, bucket, s3_key)
    size = os.path.getsize(local_file) / \
           (1024*1024)
    print(f"  ✅ {s3_key} "
          f"({size:.1f} MB)")

print("\n✅ All processed data in S3")

# Verify S3 contents
response = s3_client.list_objects_v2(
    Bucket=bucket, Prefix='processed/')
print("\n=== S3 PROCESSED FOLDER ===")
for obj in response.get('Contents', []):
    size_mb = obj['Size'] / (1024*1024)
    print(f"  {obj['Key']:<40} "
          f"{size_mb:.1f} MB")

✅ Saved locally

Uploading to S3...
  ✅ processed/train_tensors.pt (211.8 MB)
  ✅ processed/val_tensors.pt (23.5 MB)
  ✅ processed/test_tensors.pt (14.9 MB)
  ✅ processed/prep_config.json (0.0 MB)

✅ All processed data in S3

=== S3 PROCESSED FOLDER ===
  processed/prep_config.json               0.0 MB
  processed/test_tensors.pt                14.9 MB
  processed/train_tensors.pt               211.8 MB
  processed/val_tensors.pt                 23.5 MB


In [13]:
# Cell 12 — End-to-end pipeline test
print("=== PIPELINE VERIFICATION ===\n")

# Load back from saved tensors
checkpoint = torch.load(
    '../data/processed/train_tensors.pt',
    map_location='cpu')

print(f"✅ Tensors loadable")
print(f"   input_ids:      "
      f"{checkpoint['input_ids'].shape}")
print(f"   attention_mask: "
      f"{checkpoint['attention_mask'].shape}")
print(f"   labels:         "
      f"{checkpoint['labels'].shape}")

# Decode a sample to verify correctness
sample_idx = 0
input_id_sample = checkpoint[
    'input_ids'][sample_idx]
decoded = tokenizer.decode(
    input_id_sample,
    skip_special_tokens=True)
true_label = checkpoint['labels'][
    sample_idx].item()
label_map  = {0:'World', 1:'Sports',
              2:'Business', 3:'Sci/Tech'}

print(f"\n=== SAMPLE VERIFICATION ===")
print(f"True label: {label_map[true_label]}")
print(f"Decoded:    {decoded[:150]}...")
print(f"\n✅ Encoding → Decoding verified correctly")

=== PIPELINE VERIFICATION ===

✅ Tensors loadable
   input_ids:      torch.Size([108000, 128])
   attention_mask: torch.Size([108000, 128])
   labels:         torch.Size([108000])

=== SAMPLE VERIFICATION ===
True label: Sports
Decoded:    10 seconds that change everything athens - ten seconds. barely time enough to tie a shoe, wash a glass, get the paper off the porch. but when the mome...

✅ Encoding → Decoding verified correctly
